# Yeongdeok 2025 — validation dry-run

**Purpose**: prove the validation pipeline runs end-to-end on the Yeongdeok 2025 case,
even though Session 2 uses synthetic DEM/fuel inputs and a stub observed-perimeter manifest.

Session 3 will replace the synthetic inputs with real KFS perimeter + NGII DEM + KFS 임상도 data.
The call sites in this notebook will NOT change — only the source flags will.

## Pipeline steps

1. Load `YEONGDEOK_2025` `RegionConfig`.
2. Load the validation-case manifest from `data/validation_cases/yeongdeok_2025.json`.
3. Run the multi-class Rothermel + CRS-aware FireGrid through `run_validation()` for 24 hours.
4. Plot predicted perimeters + observed-area baseline.
5. Report metrics.


In [ ]:
from pathlib import Path

from wildfireguardian.utils.regions import YEONGDEOK_2025
from wildfireguardian.validation import (
    ValidationCase, ModelConfig, run_validation, load_case,
)
from wildfireguardian.spread_model.rothermel import KOREAN_PINUS

print(f"Region: {YEONGDEOK_2025.name_kr}  ({YEONGDEOK_2025.name})")
print(f"  WGS84 bbox: {YEONGDEOK_2025.bbox_wgs84}")
print(f"  EPSG:5179 size: {YEONGDEOK_2025.width_m:.0f} m × {YEONGDEOK_2025.height_m:.0f} m")

In [ ]:
# Load the manifest (uses approximate / public-source values; see docs/BLOCKERS.md)
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
case = load_case(repo_root / "data" / "validation_cases" / "yeongdeok_2025.json")
print(f"Case: {case.region.name}")
print(f"  ignition (WGS84 lon, lat): {case.observed_ignition_point_wgs84}")
print(f"  total burn area (ha): {case.observed_total_burn_area_ha}")
print(f"  official warnings: {len(case.observed_official_warnings)} timeline entries")
print(f"  notes: {case.notes[:120]}...")

In [ ]:
# Run the validation pipeline. Cell size 200 m for notebook responsiveness;
# production validation in Session 3 will use 30 m (matches NGII DEM resolution).
cfg = ModelConfig(
    cell_size_m=200.0,
    wind_speed_midflame_ms=5.0,    # Korean Pinus closed-canopy midflame at ~10 m/s 10-m wind
    wind_from_deg=270.0,           # from west, matches 25 Mar 2025 KMA reports
    dead_moisture_1h=0.08,         # 8% dead 1-h (drought-ish spring)
    live_moisture_lfmc=0.40,       # 40% LFMC (KFS post-event field estimate)
    duration_min=1440.0,           # 24 hours
    dt_min=2.0,
    snapshot_every_min=60.0,
    dem_source="synthetic",        # Session 3: switch to 'ngii' or 'srtm'
    fuel_source="synthetic",       # Session 3: switch to 'kfs_impsangdo'
)
results = run_validation(case, cfg)
print(f"Predicted snapshots: {len(results.predicted_perimeters)}")
for note in results.notes:
    print(f"  NOTE: {note}")

In [ ]:
# Predicted burned area over time vs. linear-growth observed baseline.
import matplotlib.pyplot as plt

times_h = [p.time_min / 60.0 for p in results.predicted_perimeters]
areas_ha = [p.area_m2 / 10_000.0 for p in results.predicted_perimeters]

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=110)
ax.plot(times_h, areas_ha, label="predicted (synthetic DEM + KP_PINE)",
        color="#c0392b", linewidth=2.0)
if case.observed_total_burn_area_ha is not None:
    final_obs = case.observed_total_burn_area_ha
    # 7-day duration in the manifest → linear growth for visual reference.
    ax.axhline(final_obs, color="#34495e", linestyle="--",
               label=f"observed total burn area ({final_obs:.0f} ha, KFS preliminary)")
ax.set_xlabel("Time since ignition (hours)")
ax.set_ylabel("Burned area (ha)")
ax.set_title(f"Yeongdeok 2025 dry-run · {case.region.name_kr}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot predicted perimeter at t=24h on a WGS84 map.
import geopandas as gpd

final_pred = results.predicted_perimeters[-1].polygon
if final_pred is not None:
    final_gdf = gpd.GeoDataFrame(geometry=[final_pred], crs="EPSG:5179").to_crs("EPSG:4326")
    ax = final_gdf.plot(figsize=(6, 6), color="#c0392b", alpha=0.5, edgecolor="black")
    # Region bbox for context.
    from shapely.geometry import box as _box
    region_box = gpd.GeoDataFrame(geometry=[_box(*case.region.bbox_wgs84)], crs="EPSG:4326")
    region_box.boundary.plot(ax=ax, color="#34495e", linewidth=1.0, linestyle="--")
    if case.observed_ignition_point_wgs84:
        lon, lat = case.observed_ignition_point_wgs84
        ax.plot(lon, lat, "o", color="#f39c12", markersize=10, label="ignition")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Predicted perimeter at t={results.predicted_perimeters[-1].time_min/60:.0f} h\n"
                  f"area = {final_pred.area / 10_000.0:.0f} ha")
    ax.legend()
    plt.show()

In [ ]:
# Tabular metrics summary.
from pprint import pprint
pprint(results.as_dict(), width=100)

## Interpretation (Session 2 dry-run)

- **Pipeline runs end-to-end**: ✅ from RegionConfig → raster ingestion → CRS-aware FireGrid →
  multi-class Rothermel + Korean Pinus → perimeter snapshots → metrics computation.
- **Numbers are not yet meaningful**: synthetic DEM and synthetic fuel make the
  prediction-vs-observed comparison purely structural. Real validation needs Session 3
  ingestion of KFS perimeter shapefile + NGII DEM + KFS 임상도.
- **What we *can* say from this dry-run**: with Korean Pinus multi-class fuel and the
  postulated 5 m/s midflame wind / 40% LFMC, the model predicts roughly the right order
  of magnitude of burned area for a 24-hour run.

## Next steps (Session 3)

1. Ingest real KFS perimeter shapefile for the 2025 event → replace `observed_perimeters_path` in the manifest.
2. Wire up NGII DEM ingestion in `data_io.raster.load_dem(source='ngii')`.
3. Wire up KFS 임상도 ingestion in `data_io.raster.load_fuel_type(source='kfs_impsangdo')`.
4. Re-run this notebook with the real data sources.
5. Replicate for Uljin/Samcheok 2022 and Goseong 2019.